# Inner Speech Dataset — example data loading

The dataset contains inner speech EEG recordings in two languages: **Russian** (12 subjects, 14 words) and **Spanish** (10 subjects, 12 words).  
Available per subject:
- `*.edf` / `*_edited.edf` — raw EEG recording
- `*-epo.fif` — epochs
- `*-clean-epo.fif` — epochs after artefact removal

Each word was recorded in two blocks: **overt** (spoken aloud, suffix `1`) and **inner** (imagined, suffix `2`).

In [ ]:
import sys
sys.path.insert(0, '..')

import mne
import numpy as np
import pandas as pd
from pathlib import Path

import utils
from utils import (
    load_raw, load_all_raw,
    load_epochs, load_all_epochs, load_all_languages,
    load_labels, load_all_labels,
    LABEL_INFO, INFORMATIVE, SERVICE, ATYPICAL,
)

# Set dataset root — change this path to match your local setup
utils.DEFAULT_BASE = Path(r'D:\inner_speech_v2')

mne.set_log_level('WARNING')

## 1. Label reference

In [2]:
display(LABEL_INFO)

print(f'\nInformative labels ({len(INFORMATIVE)}): {sorted(INFORMATIVE)}')
print(f'Service labels: {sorted(SERVICE)}')

,id_russian,id_spanish,english,russian,spanish,speech_type
label,,,,,,
BACK1,1,1.0,back,назад,atrás,overt
BACK2,2,2.0,back,назад,atrás,inner
DOWN1,3,3.0,down,вниз,abajo,overt
DOWN2,4,4.0,down,вниз,abajo,inner
FORWARD1,5,5.0,forward,вперёд,adelante,overt
FORWARD2,6,6.0,forward,вперёд,adelante,inner
GO,7,7.0,go,старт,inicio,service: block start
GZ,8,8.0,rest,покой,descanso,service: rest/baseline
LEFT1,9,9.0,left,влево,izquierda,overt



Informative labels (14): ['BACK1', 'BACK2', 'DOWN1', 'DOWN2', 'FORWARD1', 'FORWARD2', 'LEFT1', 'LEFT2', 'NEXT1', 'NEXT2', 'RIGHT1', 'RIGHT2', 'UP1', 'UP2']
Service labels: ['GO', 'GZ']


## 2. Loading raw EDF

In [3]:
# Single subject
raw = load_raw('sub2', 'Russian')
print(raw)
print(f'Channels: {len(raw.ch_names)}, sfreq: {raw.info["sfreq"]} Hz, duration: {raw.times[-1]:.1f} s')

C:\Users\a4431\AppData\Roaming\Python\Python39\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


<RawEDF | sub2_session.edf, 37 x 1970500 (3941.0 s), ~40 kB, data not loaded>
Channels: 37, sfreq: 500.0 Hz, duration: 3941.0 s


In [4]:
# Annotations
ann_df = pd.DataFrame({
    'onset_s':     raw.annotations.onset,
    'duration_s':  raw.annotations.duration,
    'description': raw.annotations.description,
})

print('Unique labels:', sorted(ann_df['description'].unique()))
print(f'\nFirst 10 annotations:')
display(ann_df.head(10))

Unique labels: ['BACK1', 'BACK2', 'DOWN1', 'DOWN2', 'FORWARD1', 'FORWARD2', 'GO', 'GZ', 'LEFT1', 'LEFT2', 'NEXT1', 'NEXT2', 'RIGHT1', 'RIGHT2', 'UP1', 'UP2']

First 10 annotations:


,onset_s,duration_s,description
0,15.796,0.0,GO
1,16.796,0.0,GO
2,17.796,0.0,GO
3,19.796,0.0,GO
4,20.796,0.0,GO
5,21.796,0.0,GO
6,22.796,0.0,GO
7,23.796,0.0,GO
8,24.796,0.0,GO
9,25.796,0.0,GO


In [5]:
# All subjects for one language
all_raw_ru = load_all_raw('Russian')
print('Loaded:', list(all_raw_ru.keys()))

c:\Users\a4431\VSProjects\innersp\innersp\loader.py:81: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  result[subject] = mne.io.read_raw_edf(str(edf), preload=preload, verbose=False)


Loaded: ['sub1', 'sub10', 'sub11', 'sub12', 'sub2', 'sub3', 'sub4', 'sub5', 'sub6', 'sub7', 'sub8', 'sub9']


## 3. Loading epochs (FIF)

In [6]:
# Single subject — clean epochs
epochs = load_epochs('sub2', 'Russian')
print(epochs)
print(f'\nEpochs: {len(epochs)}')
print(f'Channels: {len(epochs.ch_names)}')
print(f'Time window: [{epochs.tmin:.3f}, {epochs.tmax:.3f}] s')
print(f'Labels: {epochs.event_id}')

<EpochsFIF | 1485 events (all good), -0.5 – 0.998 s (baseline off), ~59 kB, data not loaded,
 'BACK1': 84
 'BACK2': 87
 'DOWN1': 83
 'DOWN2': 84
 'FORWARD1': 83
 'FORWARD2': 82
 'GO': 129
 'GZ': 148
 'LEFT1': 86
 'LEFT2': 84
 and 6 more events ...>

Epochs: 1485
Channels: 37
Time window: [-0.500, 0.998] s
Labels: {'BACK1': 1, 'BACK2': 2, 'DOWN1': 3, 'DOWN2': 4, 'FORWARD1': 5, 'FORWARD2': 6, 'GO': 7, 'GZ': 8, 'LEFT1': 9, 'LEFT2': 10, 'NEXT1': 11, 'NEXT2': 12, 'RIGHT1': 13, 'RIGHT2': 14, 'UP1': 15, 'UP2': 16}


c:\Users\a4431\VSProjects\innersp\innersp\loader.py:96: RuntimeWarning: This filename (D:\inner_speech_v2\preprocessed\russian\sub2\sub2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  return mne.read_epochs(str(path), preload=preload, verbose=False)


In [7]:
# Filter by condition: inner speech only
inner_labels = [l for l in epochs.event_id if l.endswith('2') and l not in SERVICE]
epochs_inner = epochs[inner_labels]
print(f'Inner speech epochs: {len(epochs_inner)}')

# Overt speech only
overt_labels = [l for l in epochs.event_id if l.endswith('1') and l not in SERVICE]
epochs_overt = epochs[overt_labels]
print(f'Overt speech epochs: {len(epochs_overt)}')

Inner speech epochs: 595
Overt speech epochs: 613


In [8]:
# All subjects across both languages
all_epochs = load_all_languages()

rows = []
for language, subjects in all_epochs.items():
    for subject, ep in subjects.items():
        is_atypical = (language, subject) in ATYPICAL
        rows.append({
            'language': language,
            'subject':  subject,
            'n_epochs': len(ep),
            'n_channels': len(ep.ch_names),
            'sfreq_Hz': ep.info['sfreq'],
            'tmin_s':   round(ep.tmin, 3),
            'tmax_s':   round(ep.tmax, 3),
            'atypical': '⚠' if is_atypical else '',
        })

display(pd.DataFrame(rows).set_index(['language', 'subject']))

c:\Users\a4431\VSProjects\innersp\innersp\loader.py:109: RuntimeWarning: This filename (D:\inner_speech_v2\preprocessed\russian\sub1\sub1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  result[subject] = mne.read_epochs(str(fif), preload=preload, verbose=False)
c:\Users\a4431\VSProjects\innersp\innersp\loader.py:109: RuntimeWarning: This filename (D:\inner_speech_v2\preprocessed\russian\sub10\sub10_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  result[subject] = mne.read_epochs(str(fif), preload=preload, verbose=False)
c:\Users\a4431\VSProjects\innersp\innersp\loader.py:109: RuntimeWarning: This filename (D:\inner_speech_v2\preprocessed\russian\sub11\sub11_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  result[subject] 

n_epochs  n_channels  sfreq_Hz  tmin_s  tmax_s atypical
language subject                                                         
Russian  sub1          864          37     500.0     0.0   0.998        ⚠
         sub10         912          37     500.0     0.0   0.998        ⚠
         sub11        1427          40     500.0    -0.5   0.998         
         sub12        1492          37     500.0    -0.5   0.998         
         sub2         1485          37     500.0    -0.5   0.998         
         sub3          912          37     500.0     0.0   0.998        ⚠
         sub4         1492          37     500.0    -0.5   0.998         
         sub5          804          37     500.0     0.0   0.998        ⚠
         sub6         1449          37     500.0    -0.5   0.998        ⚠
         sub7         2405          37     500.0    -0.5   0.998         
         sub8         1481          37     500.0    -0.5   0.998         
         sub9         2850          35     250.0    -0.5   0.996         
Spanish  sub0         2029          38     500.0    -0.5   0.998         
         sub1         1631          38     500.0    -0.5   0.998         
         sub2         2208          38     500.0    -0.5   0.998         
         sub3         1101          38     500.0    -0.5   0.998         
         sub4         1168          38     500.0    -0.5   0.998         
         sub5         1657          38     500.0    -0.5   0.998         
         sub6         1552          40     500.0    -0.5   0.998         
         sub7         1620          40     500.0    -0.5   0.998         
         sub8         1425          40     500.0    -0.5   0.998         
         sub9         1267          40     500.0    -0.5   0.998

## 4. Pre-stimulus overlap check

Verify that the 0.5 s pre-stimulus window does not overlap with the previous marker's activity (assumed to last 300 ms).

In [ ]:
TMIN      = 0.5  # pre-stimulus window used during epoching (s)
POST_PREV = 0.3  # activity assumed to persist after previous marker (s)
MIN_SAFE  = TMIN + POST_PREV  # minimum safe inter-marker gap = 0.8 s

rows = []
for language in ('Russian', 'Spanish'):
    for edf in sorted((utils.DEFAULT_BASE / 'raw' / language.lower()).rglob('*_session.edf')):
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)

        word_onsets = np.array([
            o for o, d in zip(raw.annotations.onset, raw.annotations.description)
            if d.upper() not in SERVICE
        ])
        if len(word_onsets) < 2:
            continue

        word_onsets.sort()
        gaps    = np.diff(word_onsets)
        min_gap = float(gaps.min())
        n_bad   = int((gaps < MIN_SAFE).sum())

        rows.append({
            'language':  language,
            'subject':   subject,
            'min_gap_s': round(min_gap, 3),
            'n_overlap': n_bad,
            'ok':        'ok' if n_bad == 0 else f'! {n_bad} pairs',
        })

df_overlap = pd.DataFrame(rows).sort_values(['language', 'subject'])
display(df_overlap.set_index(['language', 'subject']))
print(f'\nThreshold: {MIN_SAFE} s  (tmin={TMIN} + post_prev={POST_PREV})')
print(f'Subjects with overlap issues: {(df_overlap["n_overlap"] > 0).sum()} / {len(df_overlap)}')